In [2]:
%pip install xarray netCDF4 scipy highspy

Defaulting to user installation because normal site-packages is not writeable
  Using cached xarray-2026.9.0-py3-none-any.whl.metadata (12 kB)
  Using cached netcdf4-1.7.5.1.post1-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 2/4 [netCDF4]  WARNING: The scripts nc3tonc4, nc4tonc3 and ncinfo are installed in '/srv/home/tkha0027/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [xarray]2m3/4 [xarray]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import importlib.util
import time
from pathlib import Path
import numpy as np

# 1. Automatically locate the extracted folder containing scripts/helpers.py
matches = list(Path.home().rglob("scripts/helpers.py"))
assert matches, "Could not find scripts/helpers.py in your home directory—verify the zip finished extracting."
BASE_DIR = matches[0].parent.parent
print("Using benchmark directory:", BASE_DIR)

# 2. Load the local scripts/helpers.py directly by file path
spec = importlib.util.spec_from_file_location("luto_helpers", BASE_DIR / "scripts" / "helpers.py")
helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helpers)

# 3. Load and verify RES5
t0 = time.perf_counter()
model = helpers.load_model(BASE_DIR / "data" / "model_RES5_2045_2050")
print(f"Loaded RES5 in {time.perf_counter() - t0:.2f} s")
print("Metadata:", model.meta)

x_ref = np.load(model.dir / "x_reference.npy").astype(float)
print("Reference check:", helpers.check_solution(model, x_ref))

Using benchmark directory: /srv/home/tkha0027/luto_benchmark/LUTO_model_matix
Loaded RES5 in 7.33 s
Metadata: {'model': 'LUTO2 (Land-Use Trade-Offs model v2), one time step as a linear program', 'base_year': 2045, 'target_year': 2050, 'resfactor': 5, 'exported': '2026-11-11 11:13:07', 'objective_sense': 'maximize', 'objective_unit': 'million AUD', 'n_rows': 3810512, 'n_cols': 7085801, 'n_nonzeros': 25630105, 'n_cells': 187755, 'matrix_dtype': 'float32', 'matrix_abs_min': 0.00010035490413429216, 'matrix_abs_max': 19144.37109375, 'columns_by_block': {'ag': 1258469, 'ag2ag': 2456619, 'ag2nonag': 1187670, 'am': 1628587, 'nonag': 553379, 'nonag2ag': 1077}, 'rows_by_family': {'GBF4_ECNES': {'rows': 6, 'nonzeros': 10577, 'unit': 'ha', 'meaning': 'area of a threatened ecological community meets its target', 'structural': False}, 'GBF4_SNES': {'rows': 23, 'nonzeros': 51310, 'unit': 'ha', 'meaning': 'habitat area of a threatened species meets its target', 'structural': False}, 'ag_mgt_adoption':

In [4]:
import resource
import gurobipy as gp

def peak_ram_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)

A_csr = model.A.tocsr()
obj = np.asarray(model.cols["obj"].values, dtype=np.float64)
lb = np.asarray(model.cols["lb"].values, dtype=np.float64)
ub = np.asarray(model.cols["ub"].values, dtype=np.float64)
rhs = np.asarray(model.rows["rhs"].values, dtype=np.float64)
sense = np.asarray(model.rows["sense"].values, dtype=np.int8)

t_build = time.perf_counter()
with gp.Env() as env, gp.Model("luto_res5", env=env) as m:
    m.Params.Method = 2            # Barrier (same algorithm LUTO uses)
    m.Params.Crossover = 0         # Skip crossover for faster LP solve
    m.Params.FeasibilityTol = 1e-6
    m.Params.BarConvTol = 1e-5

    x = m.addMVar(shape=A_csr.shape[1], lb=lb, ub=ub, obj=obj, name="x")
    m.ModelSense = gp.GRB.MAXIMIZE

    for s_val, gp_sense in [(-1, gp.GRB.LESS_EQUAL), (0, gp.GRB.EQUAL), (1, gp.GRB.GREATER_EQUAL)]:
        idx = np.where(sense == s_val)[0]
        if len(idx) > 0:
            m.addMConstr(A_csr[idx, :], x, gp_sense, rhs[idx])

    m.update()
    print(f"Gurobi build time: {time.perf_counter() - t_build:.2f} s | Peak RAM: {peak_ram_gb():.2f} GB")

    t_solve = time.perf_counter()
    m.optimize()
    print(f"Gurobi solve time: {time.perf_counter() - t_solve:.2f} s | Status: {m.Status} | Peak RAM: {peak_ram_gb():.2f} GB")
    x_sol = x.X.copy()
    
print("\nSolved check:", helpers.check_solution(model, x_sol))
print("Reference obj :", model.meta["reference_solution"]["objective_of_x_reference"])

Set parameter TokenServer to value "118.138.243.190"
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter FeasibilityTol to value 1e-06
Set parameter BarConvTol to value 1e-05
Gurobi build time: 22.86 s | Peak RAM: 5.08 GB
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: AMD EPYC-Rome Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 32 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
Method  2
BarConvTol  1e-05
Crossover  0

Optimize a model with 3810512 rows, 7085801 columns and 25630105 nonzeros (Max)
Model fingerprint: 0x2dae5f69
Model has 6805998 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 2e+04]
  Objective range  [1e-04, 3e+02]
  Bounds range     [3e-04, 1e+00]
  RHS range        [2e-03, 5e+04]
Presolve removed 364952 rows and 482556 columns (presolve time = 6s)...
Presolve removed 374453 rows and 490000 columns (presolve time